# LeetCode #1130: Minimum Cost Tree From Leaf Values

https://leetcode.com/problems/minimum-cost-tree-from-leaf-values/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Interval DP)** | $O(n^3)$ | $O(n^2)$ |
| **Optimal: Monotonic Stack ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force (Interval DP)
For every contiguous subarray of leaves, try all split points to form a subtree, tracking the minimum cost. The $O(n^3)$ interval DP is correct but quadratic in both time and space compared to the stack approach.

### Optimal: Monotonic Stack ★
Maintain a decreasing monotonic stack of leaf values. Whenever a new value is larger than the stack top, the top is the local minimum and must be paired with the smaller of its two neighbours to minimise cost. Each leaf is pushed and popped exactly once, achieving $O(n)$ time and $O(n)$ space.

**Why this is better than Interval DP:** The stack processes each element once rather than re-evaluating all subranges, cutting time from $O(n^3)$ to $O(n)$.

**Constraints:**
* $2 \leq arr.length \leq 40$
* $1 \leq arr[i] \leq 15$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;
public class Solution {
    public int MctFromLeafValues(int[] arr) {
        // Decreasing stack: pair a local min with the smaller of its neighbours
        var stack = new Stack<int>();
        stack.Push(int.MaxValue); // sentinel so we never pop an empty stack
        int cost = 0;
        foreach (int val in arr) {
            while (stack.Peek() <= val) {
                // The top is a local min — its cheapest pairing is with the smaller neighbour
                int mid = stack.Pop();
                cost += mid * Math.Min(stack.Peek(), val);
            }
            stack.Push(val);
        }
        // Drain remaining elements; each pairs with its left neighbour
        while (stack.Count > 2) {
            int mid = stack.Pop();
            cost += mid * stack.Peek();
        }
        return cost;
    }
}

### Python

In [ ]:
class Solution:
    def mct_from_leaf_values(self, arr: list[int]) -> int:
        # Decreasing stack: pair a local min with the smaller of its neighbours
        stack = [float('inf')]  # sentinel so we never pop an empty stack
        cost = 0
        for val in arr:
            while stack[-1] <= val:
                # The top is a local min — its cheapest pairing is with the smaller neighbour
                mid = stack.pop()
                cost += mid * min(stack[-1], val)
            stack.append(val)
        # Drain remaining elements; each pairs with its left neighbour
        while len(stack) > 2:
            mid = stack.pop()
            cost += mid * stack[-1]
        return cost

### Go

In [ ]:
func mctFromLeafValues(arr []int) int {
    // Decreasing stack: pair a local min with the smaller of its neighbours
    stack := []int{1<<31 - 1} // sentinel so we never access an empty stack
    cost := 0
    for _, val := range arr {
        for stack[len(stack)-1] <= val {
            // The top is a local min — its cheapest pairing is with the smaller neighbour
            mid := stack[len(stack)-1]
            stack = stack[:len(stack)-1]
            left := stack[len(stack)-1]
            if left < val { cost += mid * left } else { cost += mid * val }
        }
        stack = append(stack, val)
    }
    // Drain remaining elements; each pairs with its left neighbour
    for len(stack) > 2 {
        mid := stack[len(stack)-1]
        stack = stack[:len(stack)-1]
        cost += mid * stack[len(stack)-1]
    }
    return cost
}

### Rust

In [ ]:
impl Solution {
    pub fn mct_from_leaf_values(arr: Vec<i32>) -> i32 {
        // Decreasing stack: pair a local min with the smaller of its neighbours
        let mut stack: Vec<i32> = vec![i32::MAX]; // sentinel so we never pop empty
        let mut cost = 0i32;
        for val in arr {
            while *stack.last().unwrap() <= val {
                // The top is a local min — its cheapest pairing is with the smaller neighbour
                let mid = stack.pop().unwrap();
                cost += mid * (*stack.last().unwrap()).min(val);
            }
            stack.push(val);
        }
        // Drain remaining elements; each pairs with its left neighbour
        while stack.len() > 2 {
            let mid = stack.pop().unwrap();
            cost += mid * *stack.last().unwrap();
        }
        cost
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `arr = [6, 2, 4]`
Stack processes 6 (push), then 2 (push, since 2 < 6), then 4: pops 2 (local min), pairs with `min(6, 4) = 4`, cost = 8. Drains: pops 4, pairs with 6, cost += 24. Total: **32**.

### 2. Slightly Complex
**Input:** `arr = [3, 1, 5, 8]`
Stack: push 3, push 1 (smaller), encounter 5 → pop 1 (pairs with min(3,5)=3), cost=3. Push 5, encounter 8 → pop 5 (pairs with min(3,8)=3? no, stack top is 3), wait — pop 5 pairs with min(3,8)=3, cost+=15. Drain: pop 3, pair with 8, cost+=24. Wait, let me re-check: After popping 1, stack=[3], push 5. After encounter 8: pop 5 (pairs with min(3,8)=3), cost=3+15=18. Push 8. Drain: pop 8 pairs with 3, cost=18+24=42. Answer: **42**.

### 3. Edge Case: Time Factor
**Input:** `arr = [1, 2, 3, ..., 40]` (strictly increasing).
Each new value immediately pops the stack top (larger value encountered). Every element is pushed once and popped once — exactly $2 \times 40 = 80$ operations total, confirming $O(n)$.

### 4. Edge Case: Space Factor
**Input:** `arr = [15, 14, 13, ..., 1]` (strictly decreasing).
All 40 values are pushed before any pop, filling the stack to its maximum depth of 41 (40 values + sentinel). The drain phase then pops all 40 elements sequentially.

### 5. Almost-Impossible but Plausible
**Input:** `arr = [1, 15, 1, 15, 1]` — alternating min and max.
Each 1 is sandwiched between 15s. The stack pops 1 and always pairs it with `min(15, 15) = 15`, cost $= 15$ per pop. Two 1s are popped this way ($cost = 30$), then the remaining 1 is drained against a 15 ($cost = 15$), total: **45**.